# Setup/Imports

In [ ]:
import sys
import os

import torch

sys.path.append(os.path.abspath("../src"))

from models import SimpleCNN, ResNet18, DenseNet121, EfficientNetB0, MobileNetV2, ShuffleNetV2, SqueezeNet

from data import (
    prepare_full_dataframe, 
    prepare_data
 )

from utils import (
    get_device,
    load_experiment_outputs,
    save_experiment_outputs,
    get_experiment_outputs_path
 )

from train import run_training_pipeline, run_smoke_test

from experiement_types import Metrics, History, ModelOutput

import config

In [ ]:
import sys
print(f"Python executable: {sys.executable}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


# Data Preprocessing

In [ ]:
dataset_path = config.DATASET_PATH
print("Dataset location:", dataset_path)

metadata_file = os.path.join(dataset_path, "Data_Entry_2017.csv")
df = prepare_full_dataframe(metadata_file, dataset_path)

print("Total images:", len(df))
print("Unique patients:", df["Patient ID"].nunique())
df["Finding Labels"].value_counts().head()

In [ ]:
print(df["split"].value_counts())

In [ ]:
train_loader, val_loader, test_loader = prepare_data(df)
device = get_device()

# Model Individual Training/Testing

## Training Set Up

In [ ]:
model_registry = {
    "SimpleCNN": SimpleCNN,
    "ResNet18": lambda: ResNet18(num_classes=2, in_channels=1),
    "DenseNet121": lambda: DenseNet121(num_classes=2, in_channels=1),
    "EfficientNet-B0": lambda: EfficientNetB0(num_classes=2, in_channels=1),
    "MobileNetV2": lambda: MobileNetV2(num_classes=2, in_channels=1),
    "ShuffleNetV2": lambda: ShuffleNetV2(num_classes=2, in_channels=1),
    "SqueezeNet": lambda: SqueezeNet(num_classes=2, in_channels=1)
}
model_names = list(model_registry.keys())
model_builders = list(model_registry.values())

experiment_outputs_path = get_experiment_outputs_path()
experiment_outputs = load_experiment_outputs(experiment_outputs_path)
print(f"Loaded {len(experiment_outputs)} saved result(s) from {experiment_outputs_path}")

def _print_training_summary(metrics, history):
    print("\n=== Training Summary ===")
    print(f"Model: {metrics['model']}")
    print(f"Epochs: {metrics['epochs']}")
    print(
        "Test Metrics -> "
        f"Loss: {metrics['test_loss']:.4f}, "
        f"Acc: {metrics['accuracy']:.4f}, "
        f"Precision: {metrics['precision']:.4f}, "
        f"Recall: {metrics['recall']:.4f}, "
        f"F1: {metrics['f1']:.4f}, "
        f"AUPRC: {metrics['auprc']:.4f}"
    )
    best_epoch = history.get("best_epoch")
    if best_epoch is not None:
        print(f"Best Epoch: {best_epoch}")
    lr_backbone = history.get("lr_backbone", [])
    lr_head = history.get("lr_head", [])
    if lr_backbone or lr_head:
        if lr_backbone:
            print(f"Backbone LR (last): {lr_backbone[-1]:.6g}")
        if lr_head:
            print(f"Head LR (last): {lr_head[-1]:.6g}")
    elif history.get("lr"):
        print(f"LR (last): {history['lr'][-1]:.6g}")

def train_and_store_model(index, live_plot=True, return_output=False):
    metrics, history = run_training_pipeline(
        model_names[index],
        model_builders[index],
        train_loader,
        val_loader,
        test_loader,
        device,
        live_plot=live_plot,
    )
    model_output = ModelOutput(
        metrics=Metrics(**metrics),
        history=History(**history),
    )
    experiment_outputs[model_names[index]] = model_output
    save_experiment_outputs(experiment_outputs, experiment_outputs_path)

    print(f"Saved {model_names[index]} results to {experiment_outputs_path}")
    _print_training_summary(metrics, history)

    if return_output:
        return model_output
    return None

## Simple CNN

### Smoke Test

In [ ]:
run_smoke_test(
    model_name=model_names[0],
    model_builder=model_builders[0],
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    device=device,
    live_plot=False,
    persist_outputs=True,
    persist_model_name=f"{model_names[0]}-SMOKE",
)

### Training

In [ ]:
train_and_store_model(0, live_plot=True)

## ResNet-18

### Smoke Test

In [ ]:
run_smoke_test(
    model_name=model_names[1],
    model_builder=model_builders[1],
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    device=device,
    live_plot=False,
    persist_outputs=True,
    persist_model_name=f"{model_names[1]}-SMOKE",
)

### Training

In [ ]:
train_and_store_model(1, live_plot=True)

## DenseNet-121

### Smoke Test

In [ ]:
run_smoke_test(
    model_name=model_names[2],
    model_builder=model_builders[2],
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    device=device,
    live_plot=False,
    persist_outputs=True,
    persist_model_name=f"{model_names[2]}-SMOKE",
)

### Training

In [ ]:
train_and_store_model(2, live_plot=True)

## EfficientNet-B0

### Smoke Test

In [ ]:
run_smoke_test(
    model_name=model_names[3],
    model_builder=model_builders[3],
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    device=device,
    live_plot=False,
    persist_outputs=True,
    persist_model_name=f"{model_names[3]}-SMOKE",
)

### Training

In [ ]:
train_and_store_model(3, live_plot=True)

## MobileNet-V2

### Smoke Test

In [ ]:
run_smoke_test(
    model_name=model_names[4],
    model_builder=model_builders[4],
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    device=device,
    live_plot=False,
    persist_outputs=True,
    persist_model_name=f"{model_names[4]}-SMOKE",
)

### Training

In [ ]:
train_and_store_model(4, live_plot=True)

## ShuffleNetV2

### Smoke Test

In [ ]:
run_smoke_test(
    model_name=model_names[5],
    model_builder=model_builders[5],
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    device=device,
    live_plot=False,
    persist_outputs=True,
    persist_model_name=f"{model_names[5]}-SMOKE",
)

### Training

In [ ]:
train_and_store_model(5, live_plot=True)

## SqueezeNet

### Smoke Test

In [ ]:
run_smoke_test(
    model_name=model_names[6],
    model_builder=model_builders[6],
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    device=device,
    live_plot=False,
    persist_outputs=True,
    persist_model_name=f"{model_names[6]}-SMOKE",
)

### Training

In [ ]:
train_and_store_model(6, live_plot=True)

# Model Comparison Plots
Visualize the test results for all models using bar plots for accuracy, precision, recall, and F1 score.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

experiment_outputs = load_experiment_outputs(experiment_outputs_path)
if not experiment_outputs:
    raise ValueError(
        f"No saved experiment outputs found at {experiment_outputs_path}. Run at least one training cell first."
    )

results_df = pd.DataFrame([
    vars(out.metrics) for out in experiment_outputs.values()
])

metrics = ["accuracy", "precision", "recall", "f1", "auprc"]

plt.figure(figsize=(12, 6))
results_melted = results_df.melt(
    id_vars="model",
    value_vars=metrics,
    var_name="Metric",
    value_name="Score"
 )
sns.barplot(data=results_melted, x="model", y="Score", hue="Metric")
plt.title("Model Comparison on Test Set")
plt.ylabel("Score")

min_score = results_melted["Score"].min()
max_score = results_melted["Score"].max()
padding = max(0.01, 0.08 * (max_score - min_score))
ymin = max(0.0, min_score - padding)
ymax = min(1.0, max_score + padding)
plt.ylim(ymin, ymax)

plt.legend(loc="lower right")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()